### Merge address file with checkpoint file

In [1]:
import pandas as pd
import os

In [2]:
notebook_dir = os.getcwd()

In [3]:
md_file_path = os.path.join(notebook_dir, '..', 'data', 'missing_dates_csv')

In [6]:
geocoded_csv_path = os.path.join('/Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/scripts/geocode_cache_t.csv')
geocoded = pd.read_csv(geocoded_csv_path)

checkpoint2_csv_path = os.path.join('/Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/scripts/data/missing_dates_csv/md_checkpoints/checkpoint2.csv')
checkpoint2 = pd.read_csv(checkpoint2_csv_path)

/var/folders/f6/0w5q0_md1413229qftcy9h840000gn/T/ipykernel_8585/2221789132.py:5: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  checkpoint2 = pd.read_csv(checkpoint2_csv_path)


In [15]:
# clean old long lat logic
checkpoint2_copy = checkpoint2.copy()

checkpoint2_copy = checkpoint2_copy.drop(
    columns=[
        "latitude",
        "longitude",
        "geocode_confidence",
        "raw_address"
    ],
    errors="ignore"
)

checkpoint2_copy["raw_address"] = (
    checkpoint2_copy["Location"]
    .astype(str)
    .str.strip()
)

In [16]:
checkpoint2_copy = checkpoint2.copy()

In [17]:
#gffi

# Clean keys before merge
checkpoint2_copy["raw_address"] = checkpoint2_copy["Location"].astype(str).str.strip()
geocoded["raw_address"] = geocoded["raw_address"].astype(str).str.strip()

# Keep only needed cache columns
geocoded_merge = geocoded[
    ["raw_address", "lat", "long", "address_confidence"]
].drop_duplicates(subset="raw_address", keep="last")

# Merge instead of iterrows
checkpoint3 = checkpoint2_copy.merge(
    geocoded_merge,
    on="raw_address",
    how="left"
)

# Rename to final column names
checkpoint3 = checkpoint3.rename(columns={
    "lat": "latitude",
    "long": "longitude",
    "address_confidence": "geocode_confidence"
})

output_path = "/Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/scripts/data/missing_dates_csv/md_checkpoints/checkpoint3_geocoded.csv"

checkpoint3.to_csv(output_path, index=False)

print(f"Saved: {output_path}")
print(f"Rows: {len(checkpoint3):,}")
print(f"Rows with coordinates: {checkpoint3[['latitude', 'longitude']].notna().all(axis=1).sum():,}")
print(f"Rows missing coordinates: {checkpoint3[['latitude', 'longitude']].isna().any(axis=1).sum():,}")

Saved: /Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/scripts/data/missing_dates_csv/md_checkpoints/checkpoint3_geocoded.csv
Rows: 428,527
Rows with coordinates: 428,494
Rows missing coordinates: 33


In [18]:
# Print head to verify
checkpoint3.head()

,Incident #,Date,Type,Location,Arrested,Source PDF,Location Prefix,Name,DOB,Charges,raw_address,latitude,longitude,geocode_confidence
0,18000001.0,2018-01-01 00:01:14,NOISE ORD,3 HARRIMAN ST,Yes,NaN,NaN,"AQUINO , ALFREDO",07/03/1974,A&B ON FAMILY / HOUSEHOLD MEMBER / INTIMATE PA...,3 HARRIMAN ST,42.718522,-71.148148,10.0
1,18000002.0,2018-01-01 00:08:38,LOUD NOISE,1 HARRIMAN ST FL 2,Yes,NaN,NaN,"SITHOLE , DAVID",01/21/1979,A&B ON FAMILY / HOUSEHOLD MEMBER / INTIMATE PA...,1 HARRIMAN ST FL 2,42.718522,-71.148148,10.0
2,18000003.0,2018-01-01 00:11:17,ALARM/BURG,16 ALLEN ST,No,NaN,MATOS,NaN,NaN,NaN,16 ALLEN ST,42.710782,-71.151911,10.0
3,18000004.0,2018-01-01 00:14:53,DISORDERLY,11 SUMMER ST,No,NaN,NaN,NaN,NaN,NaN,11 SUMMER ST,42.711117,-71.153015,10.0
4,18000005.0,2018-01-01 00:27:36,EXTRA SURVEIL,57 SPRINGFIELD ST,Yes,NaN,WARD SIX CLUB,"NUNEZ , MARTIN",06/26/2002,A&B DOMESTIC NO 209A IN EFFECT,57 SPRINGFIELD ST,42.699340,-71.156938,10.0


In [19]:
#gffi

#testing

checkpoint3_path = "/Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/scripts/data/missing_dates_csv/md_checkpoints/checkpoint3_geocoded.csv"

checkpoint3 = pd.read_csv(checkpoint3_path)

print("Rows:", len(checkpoint3))
print("Columns:", checkpoint3.columns.tolist())

# 2. Check required hash columns exist
required_hash_cols = ["Name", "DOB", "Charges"]
print("Missing hash columns:", [c for c in required_hash_cols if c not in checkpoint3.columns])

# 3. Check geocode columns exist
required_geo_cols = ["latitude", "longitude", "geocode_confidence"]
print("Missing geo columns:", [c for c in required_geo_cols if c not in checkpoint3.columns])

# 4. Check coordinate coverage
print("Rows with coordinates:", checkpoint3[["latitude", "longitude"]].notna().all(axis=1).sum())
print("Rows missing coordinates:", checkpoint3[["latitude", "longitude"]].isna().any(axis=1).sum())

Rows: 428527
Columns: ['Incident #', 'Date', 'Type', 'Location', 'Arrested', 'Source PDF', 'Location Prefix', 'Name', 'DOB', 'Charges', 'raw_address', 'latitude', 'longitude', 'geocode_confidence']
Missing hash columns: []
Missing geo columns: []
Rows with coordinates: 428494
Rows missing coordinates: 33


/var/folders/f6/0w5q0_md1413229qftcy9h840000gn/T/ipykernel_8585/3020849323.py:7: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  checkpoint3 = pd.read_csv(checkpoint3_path)


In [12]:
# Save output as new checkpoint3
output_path = os.path.join(md_file_path, 'md_checkpoints', 'md_checkpoint3_geocoded.csv')
checkpoint2_copy.to_csv(output_path, index=False)

OSError: Cannot save file into a non-existent directory: '/Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/scripts/../data/missing_dates_csv/md_checkpoints'

### Use legacy checkpoint file to update any addresses that werent file in current checkpoint file

In [53]:
notebook_dir = os.getcwd()

In [54]:
md_file_path = os.path.join(notebook_dir, '..', 'data', 'missing_dates_csv')

In [55]:
import pandas as pd
import os

cp3_csv_path = os.path.join(md_file_path, 'md_checkpoints', 'md_checkpoint3_geocoded.csv')
cp3 = pd.read_csv(cp3_csv_path)

# Load cp legacy data
cp_legacy_csv_path = os.path.join(md_file_path, 'md_checkpoints', 'md_legacy_checkpoint4.csv')
cp_legacy = pd.read_csv(cp_legacy_csv_path)

In [56]:
cp3_copy = cp3.copy()

In [57]:
cp3_copy.head()

,Incident #,Date,Type,Location,Arrested,Location Prefix,Name,DOB,Charges,latitude,longitude
0,24000977.0,2024-01-07 00:04:59,MV/BLOCKING,43 TEXAS AV,No,GOA,NaN,NaN,NaN,42.696181,-71.181345
1,24000979.0,2024-01-07 00:07:28,VIO CITY ORD,MCKINLEY AV,No,MOVED,NaN,NaN,NaN,42.686382,-71.155764
2,24000978.0,2024-01-07 00:07:41,VIO CITY ORD,247 PARK ST,No,CITED,NaN,NaN,NaN,42.716174,-71.171928
3,24000980.0,2024-01-07 00:29:26,AUTO ACC/NO PI,42 TOWER HILL ST,No,NaN,NaN,NaN,NaN,42.698194,-71.184037
4,24000981.0,2024-01-07 00:29:38,CK WELL BEING,615 BROADWAY,No,SPEEDWAY GAS STATION,NaN,NaN,NaN,42.717006,-71.177842


In [58]:
# create a dictionary for lookup from cp_legacy
cp_legacy_dict = (
    cp_legacy
    .drop_duplicates(subset=['Location'])
    .set_index('Location')[['Latitude', 'Longitude']]
    .to_dict(orient='index')
)

# iterate through cp3_copy and update missing lat/lon using cp_legacy_dict
for idx, row in cp3_copy.iterrows():
    if pd.isna(row['latitude']) or pd.isna(row['longitude']):
        location = row['Location']

        # check against the dictionary
        if location in cp_legacy_dict:
            cp3_copy.at[idx, 'latitude'] = cp_legacy_dict[location]['Latitude']
            cp3_copy.at[idx, 'longitude'] = cp_legacy_dict[location]['Longitude']

            # output to confirm the update
            print(f"[UPDATED] Row {idx} | Location: '{location}' | "
                  f"Latitude: {cp_legacy_dict[location]['Latitude']} | "
                  f"Longitude: {cp_legacy_dict[location]['Longitude']}")

[UPDATED] Row 8 | Location: 'S UNION ST & SALEM ST' | Latitude: 42.699816 | Longitude: -71.145761
[UPDATED] Row 12 | Location: 'MARKET ST & PARKER ST' | Latitude: 42.700599 | Longitude: -71.15341
[UPDATED] Row 18 | Location: 'COMMON ST & LAWRENCE ST' | Latitude: 42.70619918 | Longitude: -71.16698508
[UPDATED] Row 40 | Location: 'MERRIMACK ST & PARKER ST' | Latitude: 42.701862 | Longitude: -71.159556
[UPDATED] Row 58 | Location: 'BROADWAY & DAISY ST' | Latitude: 42.712105 | Longitude: -71.174156
[UPDATED] Row 64 | Location: 'COMMON ST & LAWRENCE ST' | Latitude: 42.70619918 | Longitude: -71.16698508
[UPDATED] Row 68 | Location: 'COMMON ST & LAWRENCE ST' | Latitude: 42.70619918 | Longitude: -71.16698508
[UPDATED] Row 70 | Location: 'MELVIN ST & OXFORD ST' | Latitude: 42.70355 | Longitude: -71.173699
[UPDATED] Row 132 | Location: 'BROADWAY & COMMON ST' | Latitude: 42.705821 | Longitude: -71.167853
[UPDATED] Row 148 | Location: 'MERRIMACK ST & S BROADWAY' | Latitude: 42.699338 | Longitude: 

### Clean Location column and match again

In [59]:
cp3_copy['Location'].value_counts()

Location
90 LOWELL ST               183
BRADFORD ST & BROADWAY      76
73 WINTHROP AV              74
700 ESSEX ST                72
205 BROADWAY                56
                          ... 
31 BERESFORD ST              1
372 MARKET ST #3             1
PARK ST & BUNKERHILL ST      1
JACKSON CT                   1
LFD / 625 HOWARD ST          1
Name: count, Length: 6540, dtype: int64

In [60]:
import re

# Clean function
def clean_address(address):
    if pd.isna(address): return ''
    address = re.sub(r'#.*$', '', address)     # remove unit info like #3
    address = re.sub(r'FL.*$', '', address)    # remove floor info like FL2
    address = re.sub(r'APT.*$', '', address)   # remove apartment info like APT 1B
    return address.strip()

In [61]:
# Step 1: create a new column in cp3_copy with cleaned locations
cp3_copy['Cleaned Location'] = cp3_copy['Location'].apply(clean_address)

# Step 2: build lookup dictionary from cp_legacy
cp_legacy_dict = (
    cp_legacy
    .drop_duplicates(subset=['Location'])
    .set_index('Location')[['Latitude', 'Longitude']]
    .to_dict(orient='index')
)

# Step 3: loop through cp3_copy and update lat/lon based
for idx, row in cp3_copy.iterrows():
    if pd.isna(row['latitude']) or pd.isna(row['longitude']):
        cleaned_location = row['Cleaned Location']
        if cleaned_location in cp_legacy_dict:
            cp3_copy.at[idx, 'latitude'] = cp_legacy_dict[cleaned_location]['Latitude']
            cp3_copy.at[idx, 'longitude'] = cp_legacy_dict[cleaned_location]['Longitude']

            # debug: print update confirmation
            print(f"[UPDATED] Row {idx} | Cleaned Location: '{cleaned_location}' | "
                  f"Latitude: {cp_legacy_dict[cleaned_location]['Latitude']} | "
                  f"Longitude: {cp_legacy_dict[cleaned_location]['Longitude']}")
        # else:
        #     # debug: location not found
        #     print(f"[SKIPPED] Row {idx} | Cleaned Location: '{cleaned_location}' not in cp_legacy")


[UPDATED] Row 1374 | Cleaned Location: 'BROADWAY &' | Latitude: 42.71024624 | Longitude: -71.17264374
[UPDATED] Row 2741 | Cleaned Location: '73 HAWLEY ST' | Latitude: 42.68819295 | Longitude: -71.161933
[UPDATED] Row 2776 | Cleaned Location: '22 FAIRMONT ST' | Latitude: 42.7120375 | Longitude: -71.17209189
[UPDATED] Row 2781 | Cleaned Location: '233 JACKSON ST' | Latitude: 42.7156485 | Longitude: -71.15761233
[UPDATED] Row 2784 | Cleaned Location: '86 OXFORD ST' | Latitude: 42.70448375 | Longitude: -71.17298426
[UPDATED] Row 2785 | Cleaned Location: '202 WILLOW ST' | Latitude: 42.7208699 | Longitude: -71.1733191
[UPDATED] Row 2789 | Cleaned Location: '675 ESSEX ST' | Latitude: 42.70323325 | Longitude: -71.17252342
[UPDATED] Row 2791 | Cleaned Location: '233 JACKSON ST' | Latitude: 42.7156485 | Longitude: -71.15761233
[UPDATED] Row 2801 | Cleaned Location: '49 EXCHANGE ST' | Latitude: 42.715733 | Longitude: -71.17172402
[UPDATED] Row 2812 | Cleaned Location: '23 HAWLEY ST' | Latitude: 

In [62]:
# verify
cp3_copy.head()

,Incident #,Date,Type,Location,Arrested,Location Prefix,Name,DOB,Charges,latitude,longitude,Cleaned Location
0,24000977.0,2024-01-07 00:04:59,MV/BLOCKING,43 TEXAS AV,No,GOA,NaN,NaN,NaN,42.696181,-71.181345,43 TEXAS AV
1,24000979.0,2024-01-07 00:07:28,VIO CITY ORD,MCKINLEY AV,No,MOVED,NaN,NaN,NaN,42.686382,-71.155764,MCKINLEY AV
2,24000978.0,2024-01-07 00:07:41,VIO CITY ORD,247 PARK ST,No,CITED,NaN,NaN,NaN,42.716174,-71.171928,247 PARK ST
3,24000980.0,2024-01-07 00:29:26,AUTO ACC/NO PI,42 TOWER HILL ST,No,NaN,NaN,NaN,NaN,42.698194,-71.184037,42 TOWER HILL ST
4,24000981.0,2024-01-07 00:29:38,CK WELL BEING,615 BROADWAY,No,SPEEDWAY GAS STATION,NaN,NaN,NaN,42.717006,-71.177842,615 BROADWAY


In [63]:
print(cp3_copy['Location'].isna().sum())

0


In [64]:
# updated cp3 as cp4
output_path = os.path.join(md_file_path, 'md_checkpoints', 'md_checkpoint4_geocode_complete.csv')
cp3_copy.to_csv(output_path, index=False)

### After Geocoding & Categorizing...

### Finding if Lat and Long are within Massachusetts

In [65]:
import networkx as nx
import osmnx as ox
import pandas as pd
import os
from shapely.geometry import Point

In [66]:
notebook_dir = os.getcwd()

In [67]:
md_file_path = os.path.join(notebook_dir, '..', 'data', 'missing_dates_csv')

In [68]:
checkpoint7_path = os.path.join(md_file_path, 'md_checkpoints', 'md_checkpoint7_serious_crimes.csv')
checkpoint7_df = pd.read_csv(checkpoint7_path)


In [69]:
# Get Massachusetts boundary polygon
massachusetts = ox.geocode_to_gdf('Massachusetts, USA')
mass_polygon = massachusetts.geometry.iloc[0]

# Drop rows with missing coordinates
checkpoint7_df = checkpoint7_df.dropna(subset=['latitude', 'longitude'])

# Create Point objects for each row
checkpoint7_df['geometry'] = checkpoint7_df.apply(
    lambda row: Point(row['longitude'], row['latitude']), axis=1
)

# Filter rows where the point is inside the Massachusetts polygon
checkpoint7_df_filtered = checkpoint7_df[checkpoint7_df['geometry'].apply(lambda point: point.within(mass_polygon))]

# Drop the geometry column if no longer needed
checkpoint7_df_filtered = checkpoint7_df_filtered.drop(columns=['geometry'])

checkpoint7_df_filtered.head()

,Incident #,Date,Type,Location,Arrested,Location Prefix,DOB,Charges,latitude,longitude,Cleaned Location,person_id,category,Year,crime_severity
0,24000977.0,2024-01-07 00:04:59,MV/BLOCKING,43 TEXAS AV,No,GOA,NaN,NaN,42.696181,-71.181345,43 TEXAS AV,NaN,MOTOR_VEHICLE_INCIDENTS,2024,Non-Serious
1,24000979.0,2024-01-07 00:07:28,VIO CITY ORD,MCKINLEY AV,No,MOVED,NaN,NaN,42.686382,-71.155764,MCKINLEY AV,NaN,PUBLIC_DISTURBANCES,2024,Non-Serious
2,24000978.0,2024-01-07 00:07:41,VIO CITY ORD,247 PARK ST,No,CITED,NaN,NaN,42.716174,-71.171928,247 PARK ST,NaN,PUBLIC_DISTURBANCES,2024,Non-Serious
3,24000980.0,2024-01-07 00:29:26,AUTO ACC/NO PI,42 TOWER HILL ST,No,NaN,NaN,NaN,42.698194,-71.184037,42 TOWER HILL ST,NaN,MOTOR_VEHICLE_INCIDENTS,2024,Non-Serious
4,24000981.0,2024-01-07 00:29:38,CK WELL BEING,615 BROADWAY,No,SPEEDWAY GAS STATION,NaN,NaN,42.717006,-71.177842,615 BROADWAY,NaN,MEDICAL_AND_WELFARE_ASSISTANCE,2024,Non-Serious


In [70]:
print("Latitude range:", checkpoint7_df_filtered['latitude'].min(), "to", checkpoint7_df_filtered['latitude'].max())
print("Longitude range:", checkpoint7_df_filtered['longitude'].min(), "to", checkpoint7_df_filtered['longitude'].max())
print()
print("Original rows:", len(checkpoint7_df))
print("Filtered rows (in MA):", len(checkpoint7_df_filtered))

Latitude range: 41.959914 to 42.735361
Longitude range: -71.444712 to -71.100213

Original rows: 12971
Filtered rows (in MA): 12845


In [71]:
checkpoint8_path = os.path.join(md_file_path, 'md_checkpoints', 'md_checkpoint8_mass_filtered.csv')
checkpoint7_df_filtered.to_csv(checkpoint8_path, index=False)
print(f"Filtered data saved to: {checkpoint8_path}")


Filtered data saved to: c:\Users\Indel\Documents\gatewayinitiative-lawrencepd\scripts\..\data\missing_dates_csv\md_checkpoints\md_checkpoint8_mass_filtered.csv
